# 🚀 Modul 04 Hands-on: Advanced Transfer Learning & Fine-Tuning

Pada hands-on ini, Anda akan mempraktikkan:
1. **The 4 Fine-Tuning Strategies**: Feature Extractor vs Last Layers vs Full Tuning vs Progressive Unfreezing.
2. **The Sabotage Lab: Catastrophic Forgetting!** Mengamati hancurnya fitur ImageNet saat LR disetel seragam terlalu tinggi ($0.01$).
3. **Discriminative Learning Rates**: Menerapkan LR kecil untuk backbone ($10^{-5}$) dan LR lebih besar untuk head ($10^{-3}$).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Menggunakan device: {device}")

## 1. The Sabotage Lab: Catastrophic Forgetting
Saat melakukan fine-tuning, jika kita menyetel learning rate besar ($0.01$) pada SELURUH layer, gradien acak dari classifier head baru akan menghancurkan bobot pre-trained ImageNet yang berharga!

In [ ]:
# Data toy kustom (5 kelas)
torch.manual_seed(42)
X_toy = torch.randn(64, 3, 64, 64).to(device)
y_toy = torch.randint(0, 5, (64,)).to(device)

# Strategi A: SABOTASE (High LR seragam di seluruh layer)
model_sabotaged = resnet18(weights=ResNet18_Weights.DEFAULT)
model_sabotaged.fc = nn.Linear(512, 5)
model_sabotaged = model_sabotaged.to(device)
opt_sabotaged = optim.Adam(model_sabotaged.parameters(), lr=0.01)  # Terlalu agresif!

# Strategi B: BENAR (Discriminative LR)
model_correct = resnet18(weights=ResNet18_Weights.DEFAULT)
model_correct.fc = nn.Linear(512, 5)
model_correct = model_correct.to(device)

# Backbone LR: 1e-5, Classifier Head LR: 1e-3
backbone_params = [p for name, p in model_correct.named_parameters() if not name.startswith("fc")]
head_params = model_correct.fc.parameters()
opt_correct = optim.Adam([
    {"params": backbone_params, "lr": 1e-5},  # Sangat berhati-hati
    {"params": head_params, "lr": 1e-3}       # Cepat beradaptasi
])

crit = nn.CrossEntropyLoss()
loss_sab, loss_cor = [], []
for ep in range(30):
    # Sabotaged step
    opt_sabotaged.zero_grad()
    l_s = crit(model_sabotaged(X_toy), y_toy)
    l_s.backward(); opt_sabotaged.step()
    loss_sab.append(l_s.item())
    
    # Correct step
    opt_correct.zero_grad()
    l_c = crit(model_correct(X_toy), y_toy)
    l_c.backward(); opt_correct.step()
    loss_cor.append(l_c.item())

plt.figure(figsize=(9, 5))
plt.plot(loss_sab, "r--", label="SABOTASE: Uniform High LR (0.01) -> Osilasi & Hilang Fitur", linewidth=2)
plt.plot(loss_cor, "g-", label="BENAR: Discriminative LR (Backbone 1e-5, Head 1e-3) -> Mulus", linewidth=2.5)
plt.title("The Sabotage Lab: Catastrophic Forgetting vs Discriminative LR", fontsize=12, fontweight="bold")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.5); plt.show()